# 03 · Combine

This notebook combines your clean tables into one table. Check the keys you join on, and save the combined table in `data/interim/`.

In [1]:
from pathlib import Path

import pandas as pd

INTERIM = Path("../data/interim")

In [2]:
shares = pd.read_parquet(INTERIM / "share_prices.parquet")
inflation = pd.read_parquet(INTERIM / "inflation.parquet")
iso = pd.read_parquet(INTERIM / "country_codes.parquet")

print(shares.shape, inflation.shape, iso.shape)

(528, 3) (2871, 4) (249, 2)


In [3]:
not_countries = shares[~shares["country_code"].isin(iso["country_code"])]
print("OECD areas that are not countries:", sorted(not_countries["country_code"].unique()))

wb_groups = inflation[~inflation["country_code"].isin(iso["country_code"])]
print("World Bank areas that are not countries:", wb_groups["country"].nunique())
sorted(wb_groups["country"].unique())[:15]

OECD areas that are not countries: ['EA20']
World Bank areas that are not countries: 46


['Africa Eastern and Southern',
 'Africa Western and Central',
 'Arab World',
 'Caribbean small states',
 'Central Europe and the Baltics',
 'Channel Islands',
 'Early-demographic dividend',
 'East Asia & Pacific',
 'East Asia & Pacific (IDA & IBRD countries)',
 'East Asia & Pacific (excluding high income)',
 'Euro area',
 'Europe & Central Asia',
 'Europe & Central Asia (IDA & IBRD countries)',
 'Europe & Central Asia (excluding high income)',
 'European Union']

In [4]:
shares = shares[shares["country_code"].isin(iso["country_code"])]
inflation = inflation[inflation["country_code"].isin(iso["country_code"])]

print(shares["country_code"].nunique(), "countries with share prices")
print(inflation["country_code"].nunique(), "countries with inflation")

47 countries with share prices
215 countries with inflation


In [5]:
missing = shares[~shares["country_code"].isin(inflation["country_code"])]
sorted(missing["country_code"].unique())

[]

In [6]:
combined = shares.merge(iso, on="country_code", how="left", validate="many_to_one")

combined = combined.merge(inflation, on=["country_code", "year"], how="left", validate="one_to_one")

print(len(shares), "rows before,", len(combined), "rows after")
combined.head()

517 rows before, 517 rows after


,country_code,year,share_price_index,alpha2,country,inflation_pct
0,AUS,2014,98.61095,AU,Australia,2.487929
1,AUS,2015,100.00000,AU,Australia,1.508368
2,AUS,2016,96.06994,AU,Australia,1.276994
3,AUS,2017,105.64780,AU,Australia,1.948643
4,AUS,2018,109.20510,AU,Australia,1.911400


In [7]:
combined = combined[["country_code", "alpha2", "country", "year", "share_price_index", "inflation_pct"]]
combined.isna().sum()

country_code         0
alpha2               0
country              0
year                 0
share_price_index    0
inflation_pct        0
dtype: int64

In [8]:
combined.to_parquet(INTERIM / "combined.parquet", index=False)
combined.shape

(517, 6)